# Agenda

## Essa semana (10/12/2025)

- Revisar Max-cut
- Revisar QAOA
- Revisar Pennylane
- Usar cuQuantum com o Pennylane

## Semana que vem (07/01/2026)

- Falar sobre cudaq
- Adaptar o código do max-cut (QAOA) para cudaq
- Ver como dividir o grafo em subgrafos
- Ver como rodar esses subgrafos em paralelo

# Instalando o Pennylane e o cuQuantum

```sh
python -m venv venv
source venv/bin/activate
pip install custatevec_cu12
pip install pennylane-lightning-gpu
```

# Max-cut

**Entrada**: um grafo não direcionado $G=(V,E)$ com pesos de arestas não negativos $w_{ij}$ (use $w_{ij}=1$ para grafos sem peso).

**Objetivo**: partitionar o conjunto de vértices $V$ em dois conjuntos $S$ e $V\setminus S$ para maximizar o peso total das arestas que cruzam a partição (o corte).


![Grafo base](graph.png)

É possível formular o problema como um QUBO criando uma relação de penalização de cada aresta $(i, j)$ que pode ser representado como

$$\text{Penalização} = 1 - (x_i + x_j - 2x_ix_j)$$

Então, se os nós estão em conjuntos diferentes temos: $x_i = 0$ e $x_j = 1$ ou $x_i = 1$ e $x_j = 0$. O que resulta em $$\text{Penalização} = 1 - (0 + 1 - 2 \times 0 \times 1) = 1 - (1) = 0$$. Se ambos estão no mesmo conjunto significa que $x_i = 0$ e $x_j = 0$ ou $x_i = 1$ e $x_j = 1$. No final temos $$\text{Penalização} = 1 - (1 + 1 - 2 \times 1 \times 1) = 1 - (2 - 2*1) = 1 - 0 = 1$$.

Então somamos a penalização de todas as arestas para formar o nosso QUBO

$$
\text{QUBO}=\sum_{(i,j)\in E} 1 - (x_i + x_j - 2x_ix_j).
$$

**nota**: na apresentação foi falado em contribuição que é outra forma de encontrar a equação, mas pensando em maximar e depois podemos converter para um problema de minimizar, mas acabei mudando para um problema de minimizar para ficar compatível com o tutorial do Pennylane e acabei me confundido na apresentação :(.

In [198]:
import sympy as sp
from IPython.display import display, Math

class Binary(sp.Symbol):
    def _eval_power(self, other):
        return self
        
# Define the binary variables
x0, x1, x2, x3 = sp.symbols('x0 x1 x2 x3 x4')

# Define the edges
edges = [(0, 1), (0, 3), (1, 2), (2, 3)]

# Initialize the QUBO expression
Q = 0  # Starting with the constant term -2

# Sum the contributions from each edge
for i, j in edges:
    Q += 1 - (sp.Symbol(f'x{i}') + sp.Symbol(f'x{j}') - 2 * sp.Symbol(f'x{i}') * sp.Symbol(f'x{j}'))

#print("Então, podemos escrever o nosso problema como o QUBO:")
display(Math(r'\text{Então, podemos escrever o nosso problema como }'))
Q

<IPython.core.display.Math object>

2*x0*x1 + 2*x0*x3 - 2*x0 + 2*x1*x2 - 2*x1 + 2*x2*x3 - 2*x2 - 2*x3 + 4

# QAOA

![https://pennylane.ai/qml/demos/tutorial_qaoa_intro](qaoa-pennylane-tutorial.png)

O **hamiltoniano de custo** mede a qualidade da solução atual. Ele é projetado para penalizar estados que não satisfazem as condições ótimas do problema, ajudando a identificar se a resposta atual é melhor que as anteriores.

O **hamiltoniano de mixer** promove a exploração do espaço de soluções. Ele ajuda a "misturar" os estados quânticos para que o algoritmo possa encontrar novos candidatos a soluções, permitindo sair de mínimos locais.

E Oara cada camada, utilizamos um otimizador clássico para determinar os parâmetros γ e $\alpha$, que controlam a evolução dos estados quânticos.

## Encontrando o hamiltoniano de custo

No caso de QUBOs, existe uma maneira de encontrar um hamiltoniano através de um Ising.

Para isso basta fazer a substituição

$$x_i = \frac{s_i + 1}{2}.$$

In [211]:
# Convert binary variables to spin variables
s0, s1, s2, s3 = sp.symbols('s0 s1 s2 s3')

# Substitute x_i in Q with s_i
Q_ising = Q.subs({x0: (s0 + 1) / 2, 
                  x1: (s1 + 1) / 2, 
                  x2: (s2 + 1) / 2, 
                 x3: (s3 + 1) / 2})

Q_ising

-s0 - s1 - s2 - s3 + 2*(s0/2 + 1/2)*(s1/2 + 1/2) + 2*(s0/2 + 1/2)*(s3/2 + 1/2) + 2*(s1/2 + 1/2)*(s2/2 + 1/2) + 2*(s2/2 + 1/2)*(s3/2 + 1/2)

Simplificando

In [212]:
H = sp.simplify(Q_ising)
H

s0*s1/2 + s0*s3/2 + s1*s2/2 + s2*s3/2 + 2

Podemos desprezar a constante porque não afeta o conjunto solução.

In [201]:
H = H - 2
H

s0*s1/2 + s0*s3/2 + s1*s2/2 + s2*s3/2

O qual pode ser escrito como

$\frac{1}{2} (s_0s_1 + s_0s_3 + s_1s_2 + s_2s_3)$

que é o nosso hamiltoniano.

## Transformando o hamiltoniano em um circuito

Para isso vamos usar a afirmação (que pode ser demostrada no futuro): Hamiltonianos de QUBO sempre são **diagonais** e **comutam**.

E por comutarem, nós podemos criar o circuito da seguinte forma

$$
\forall (q_i,q_j) \in E
$$

$$
\text{CNOT}(q_i \rightarrow q_j)\;
$$

$$
R_Z(\gamma, q_j)\;
$$

$$
\text{CNOT}(q_i \rightarrow q_j)
$$

In [ ]:
def cost_hamiltonian(gamma):
    for edge in graph:
        qml.CNOT(wires=edge)
        qml.RZ(gamma, wires=edge[1])
        qml.CNOT(wires=edge)

## Hamiltoniano de mixer

Podemos usar o hamiltoniano do artigo original que consiste em aplicar uma porta X em todos os nós.

In [ ]:
def mixer_hamiltonian(alpha):
    for wire in range(nodes):
        qml.RX(alpha, wires=wire)

## Código

https://pennylane.ai/qml/demos/tutorial_qaoa_maxcut

In [3]:
import time
import pennylane as qml
from pennylane import numpy as np

nodes = 10
graph = [(0, 1), (0, 3), (1, 2), (2, 3), (8, 9)]

def run_experiment(dev):
    def bitstring_to_int(bit_string_sample):
        return int(2 ** np.arange(len(bit_string_sample)) @ bit_string_sample[::-1])
    
    def cost_hamiltonian(gamma):
        for edge in graph:
            qml.CNOT(wires=edge)
            qml.RZ(gamma, wires=edge[1])
            qml.CNOT(wires=edge)
    
    def mixer_hamiltonian(alpha):
        for wire in range(nodes):
            qml.RX(alpha, wires=wire)
    
    @qml.set_shots(20)
    @qml.qnode(dev)
    def circuit(gammas, alphas, return_samples=False):
        # apply Hadamards to get the n qubit |+> state
        for wire in range(nodes):
            qml.Hadamard(wires=wire)
        # p instances of unitary operators
        for gamma, alpha in zip(gammas, alphas):
            cost_hamiltonian(gamma)
            mixer_hamiltonian(alpha)
    
        if return_samples:
            # sample bitstrings to obtain cuts
            return qml.sample()
        # during the optimization phase we are evaluating the objective using expval
        C = qml.sum(*(qml.Z(w1) @ qml.Z(w2) for w1, w2 in graph))
        return qml.expval(C)
        
    def objective(params):
        """Minimize the negative of the objective function C by postprocessing the QNnode output."""
        return -0.5 * (len(graph) - circuit(*params))
    
    def qaoa_maxcut(n_layers=1):
        print(f"\np={n_layers:d}")
    
        # initialize the parameters near zero
        init_params = 0.01 * np.random.rand(2, n_layers, requires_grad=True)
    
        # initialize optimizer: Adagrad works well empirically
        opt = qml.AdagradOptimizer(stepsize=0.5)
    
        # optimize parameters in objective
        params = init_params.copy()
        steps = 30
        for i in range(steps):
            params = opt.step(objective, params)
            if (i + 1) % 5 == 0:
                print(f"Objective after step {i+1:3d}: {-objective(params): .7f}")
    
        # sample 100 bitstrings by setting return_samples=True and the QNode shot count to 100
        bitstrings = qml.set_shots(circuit, shots=100)(*params, return_samples=True)
        # convert the samples bitstrings to integers
        sampled_ints = [bitstring_to_int(string) for string in bitstrings]
    
        # print optimal parameters and most frequently sampled bitstring
        counts = np.bincount(np.array(sampled_ints))
        most_freq_bit_string = np.argmax(counts)
        print(f"Optimized parameter vectors:\ngamma: {params[0]}\nalpha:  {params[1]}")
        print(f"Most frequently sampled bit string is: {most_freq_bit_string:04b}")
    
        return -objective(params), sampled_ints
    
    start_time = time.perf_counter()
    qaoa_maxcut(11)
    end_time = time.perf_counter()
    
    elapsed_time = end_time - start_time
    print(f"Function execution time: {elapsed_time:.6f} seconds")

## CPU

In [4]:
dev = qml.device("default.qubit", wires=nodes)
run_experiment(dev)


p=11
Objective after step   5:  1.3000000
Objective after step  10:  2.2000000
Objective after step  15:  3.2000000
Objective after step  20:  3.0000000
Objective after step  25:  3.5000000
Objective after step  30:  4.0000000
Optimized parameter vectors:
gamma: [ 0.68303666  0.6464144   0.03853567  0.65183592  0.04014067  0.3954556
  0.92690018  0.21324625 -0.44411108 -0.08876611 -0.42079421]
alpha:  [ 1.31434306  0.01750677  1.69124612  0.54501703 -0.17752193 -0.26673362
 -0.58243403 -1.52959742 -0.54012149 -1.55084939  0.05864367]
Most frequently sampled bit string is: 0101
Function execution time: 59.749315 seconds


# CuQuantum

Bastar trocar o `device` para o `lightning.qubit`.

In [5]:
dev = qml.device("lightning.qubit", wires=nodes)
run_experiment(dev)


p=11
Objective after step   5:  3.1000000
Objective after step  10:  1.0000000
Objective after step  15:  2.7000000
Objective after step  20:  2.9000000
Objective after step  25:  3.4000000
Objective after step  30:  3.7000000
Optimized parameter vectors:
gamma: [-1.23846221 -1.48888755 -1.65729536 -1.35410919  0.02678427  0.00885538
 -0.75769099 -1.04445765 -1.02590335 -1.45292352 -1.37991173]
alpha:  [ 0.05499895 -0.2164646  -2.04565674 -0.74348387 -1.85269772 -2.56960014
 -0.83791754  0.31885014  0.43465139  0.69336442  0.41994169]
Most frequently sampled bit string is: 0101
Function execution time: 23.392627 seconds


# Exercícios

1. Testar outros grafos.
2. Testar grafos com arestas ponderadas.
3. Testar grafos densos (várias arestas) e esparsos (poucas arestas).
4. Testar grafos com ilhas, partes densas conectadas por apenas um par de nós.
5. Testar outros mixers (XY, Grover, etc) [ref. 1][1].
6. Testar outros otimizadores além do `AdagradOptimizer` [ref. 2][2].
7. Testar outros simuladores [ref. 3][3].
8. Criar um notebook para outros problemas de combinatória [ref. 4][4].

[1]: <https://docs.pennylane.ai/en/stable/code/qml_qaoa.html#mixer-hamiltonians> "Alguns exmplos de mixers"
[2]: <https://openqaoa.entropicalabs.com/optimizers/pennylane-optimizers/> "Lista de otimizadores do pennylane" 
[3]: <https://pennylane.ai/devices> "Lista de simuladores do pennylane"
[4]: <https://arxiv.org/pdf/1811.11538> "Exemplos de como formular alguns problemas de combinatória"